# Tome 15, Chapter 12: Five projects (Python companion)

The computational starts of the projects of the chapter: random-to-top and its top-$k$ generalisation (Proposition 12.1), prime factors in a short interval (Proposition 12.2), coranks of random symmetric matrices over $\mathbb F_p$ (Proposition 12.3), and moments of random multiplicative functions over $\mathbb F_q[t]$ (Proposition 12.4). Each section ends where the project begins. The computer-science application is **segmented sieving and arithmetic in $\mathbb F_q[t]$ with integer-encoded polynomials**. Exercises first, solutions at the end. Requires `numpy` and `matplotlib`.

In [1]:
import math, numpy as np, matplotlib.pyplot as plt
from collections import Counter
rng = np.random.default_rng(12)

## Exercises

**P12.1 (★★) Shuffles.** Simulate random-to-top on $n=100$ cards and estimate, at times $t=\beta n\ln n$, the distance from uniform of the law of the position of the original bottom card, and of the number of cards never chosen. Then do the same for "random card to a uniform position among the top $k$" with $k=n/2$.

**P12.2 (★★) A short interval.** Compute $\omega(n)$ for $n\in(10^{12},10^{12}+10^6]$ with a segmented sieve, compare with the Poisson and normal models, and check the inequality of Proposition 12.2 with $y=10^3$.

**P12.3 (★★) Coranks.** Estimate the law of the corank of random symmetric matrices over $\mathbb F_3$ of size $n=40$, uniform and sparse with $d=2,5,20$, and compare $P(\text{corank}=0)$ with $\prod_{i\ge0}(1-3^{-(2i+1)})$. Check Exercise 12.3 exactly.

**P12.4 (★★) Moments over $\mathbb F_q[t]$.** Count the quadruples of Proposition 12.4 exactly for $q=2$, $n\le8$ and $q=3$, $n\le5$, and look at the ratio to $n\,q^{2n}$. Check Exercise 12.4 for $q=3$.

## Solutions

**P12.1.** Cards are numbered by their initial position from the top. A good statistic is $R$, the largest $m$ such that the bottom $m$ cards are in their original relative order: for a uniform deck $P(R\ge m)=\frac1{m!}$, while for random-to-top the cards never chosen sit at the bottom in their original order, so $R$ is at least their number (Exercise 12.1).

In [2]:
def bottom_run(deck):
    m = 1
    while m < len(deck) and deck[-m-1] < deck[-m]: m += 1
    return m
def run_stat(n, k, betas, runs=1500, m=4):
    cps = {int(b * n * math.log(n)): b for b in betas}; last = max(cps)
    hits = Counter()
    for _ in range(runs):
        deck = list(range(n))
        for s in range(1, last + 1):
            c = deck.pop(rng.integers(n)); deck.insert(rng.integers(k), c)
            if s in cps and bottom_run(deck) >= m: hits[cps[s]] += 1
    return {b: hits[b] / runs for b in betas}
n = 100; betas = [0.25, 0.5, 0.75, 1.0, 1.25, 1.5]
print('uniform deck: P(R >= 4) = 1/24 =', round(1/24, 4))
for k, name in [(1, 'random-to-top'), (n // 2, 'top-half insertion'), (n, 'random-to-random')]:
    r = run_stat(n, k, betas)
    print(f'{name:20s}', '  '.join(f'β={b}: {v:.3f}' for b, v in r.items()))

uniform deck: P(R >= 4) = 1/24 = 0.0417


random-to-top        β=0.25: 1.000  β=0.5: 0.997  β=0.75: 0.474  β=1.0: 0.085  β=1.25: 0.043  β=1.5: 0.048


top-half insertion   β=0.25: 1.000  β=0.5: 0.940  β=0.75: 0.370  β=1.0: 0.081  β=1.25: 0.039  β=1.5: 0.043


random-to-random     β=0.25: 0.084  β=0.5: 0.044  β=0.75: 0.037  β=1.0: 0.035  β=1.25: 0.042  β=1.5: 0.047


For random-to-top, $P(R\ge4)$ stays near $1$ until $\beta$ approaches $1$ and then falls to the uniform value $\frac1{24}\approx0.042$, as Proposition 12.1 and Exercise 12.1 predict. Insertion among the top half behaves almost like random-to-top for this statistic. Random-to-random forgets it almost at once, although its cutoff is at $\frac34n\ln n$: a statistic that randomises quickly says nothing about the mixing time, which is why a lower bound needs a well chosen statistic. Whether the top-$k$ shuffles have cutoff, and where, is the project; the right statistic for the lower bound is part of it.

**P12.2.** A segmented sieve: for each prime $p\le\sqrt{x+h}\approx10^6$, mark the multiples of $p$ in the window, and divide them out to find the leftover prime factor (at most one exceeds $10^6$).

In [3]:
x, h = 10**12, 10**6
N = np.arange(x + 1, x + h + 1, dtype=np.int64)
rest = N.copy(); omega = np.zeros(h, dtype=np.int64); omega_y = np.zeros(h, dtype=np.int64)
L = int(math.isqrt(x + h)) + 1
sieve = np.ones(L + 1, dtype=bool); sieve[:2] = False
for i in range(2, int(L**0.5) + 1):
    if sieve[i]: sieve[i*i::i] = False
primes = np.nonzero(sieve)[0]
y = 1000
for p in primes:
    start = (-(x + 1)) % p
    idx = np.arange(start, h, p)
    omega[idx] += 1
    if p <= y: omega_y[idx] += 1
    while idx.size:
        rest[idx] //= p
        idx = idx[rest[idx] % p == 0]
omega += rest > 1
lam = math.log(math.log(x))
cnt = np.bincount(omega)
print('mean of omega:', omega.mean().round(3), '| loglog x =', round(lam, 3), '| variance:', omega.var().round(3))
for k in range(len(cnt)):
    pois = math.exp(-lam) * lam**k / math.factorial(k)
    print(f'omega={k:2d}: observed {cnt[k]/h:.4f}  Poisson {pois:.4f}')
mu_y = sum(1/p for p in primes if p <= y); pi_y = int((primes <= y).sum())
lhs = ((omega_y - mu_y)**2).sum()
print(f'Turán–Kubilius in the window: {lhs:.0f} <= h*mu_y + 5*pi(y)^2 = {h*mu_y + 5*pi_y**2:.0f}')

mean of omega: 3.601 | loglog x = 3.319 | variance: 1.603
omega= 0: observed 0.0000  Poisson 0.0362
omega= 1: observed 0.0362  Poisson 0.1201
omega= 2: observed 0.1586  Poisson 0.1993
omega= 3: observed 0.2890  Poisson 0.2205
omega= 4: observed 0.2832  Poisson 0.1830
omega= 5: observed 0.1639  Poisson 0.1215
omega= 6: observed 0.0568  Poisson 0.0672
omega= 7: observed 0.0112  Poisson 0.0319
omega= 8: observed 0.0012  Poisson 0.0132
omega= 9: observed 0.0001  Poisson 0.0049
Turán–Kubilius in the window: 1746115 <= h*mu_y + 5*pi(y)^2 = 2339200


The mean of $\omega$ is close to $\ln\ln x+B$, where $B\approx0.2615$ is Mertens's constant, and the variance is much smaller than the Poisson value: the Poisson model with mean $\ln\ln x$ is a poor fit at this height, a well-known effect of the slow growth of $\ln\ln x$ (the Sathe–Selberg formula corrects it with an explicit factor). Proposition 12.2 holds with room to spare. The project asks how short the window can be before these laws change.

**P12.3.** Rank over $\mathbb F_p$ by Gaussian elimination.

In [4]:
def rank_mod_p(A, p):
    A = A.copy() % p; r = 0; n, m = A.shape
    for c in range(m):
        piv = next((i for i in range(r, n) if A[i, c]), None)
        if piv is None: continue
        A[[r, piv]] = A[[piv, r]]
        A[r] = (A[r] * pow(int(A[r, c]), -1, p)) % p
        nz = np.nonzero(A[:, c])[0]
        for i in nz:
            if i != r: A[i] = (A[i] - A[i, c] * A[r]) % p
        r += 1
        if r == n: break
    return r
def sym(n, p, d=None):
    if d is None: U = rng.integers(0, p, size=(n, n))
    else: U = np.where(rng.random((n, n)) < d / n, rng.integers(1, p, size=(n, n)), 0)
    A = np.triu(U); return A + np.triu(A, 1).T
p, n, runs = 3, 40, 400
limit = np.prod([1 - p**-(2*i+1) for i in range(60)])
for d in [None, 20, 5, 2]:
    cor = Counter(n - rank_mod_p(sym(n, p, d), p) for _ in range(runs))
    dist = {k: round(v / runs, 3) for k, v in sorted(cor.items())}
    print('uniform' if d is None else f'sparse d={d}', dist)
print('limit P(corank 0) for uniform:', round(limit, 4))
tot = sum(1 for a in range(p) for b in range(p) for c in range(p) if (a*c - b*b) % p == 0)
print('2x2 singular probability over F_3:', tot, '/', p**3, '= 1/p:', tot * p == p**3)

uniform {0: 0.662, 1: 0.302, 2: 0.035}


sparse d=20 {0: 0.627, 1: 0.323, 2: 0.05}


sparse d=5 {0: 0.507, 1: 0.37, 2: 0.102, 3: 0.018, 4: 0.003}


sparse d=2 {2: 0.005, 3: 0.033, 4: 0.055, 5: 0.09, 6: 0.128, 7: 0.1, 8: 0.172, 9: 0.09, 10: 0.117, 11: 0.077, 12: 0.06, 13: 0.022, 14: 0.022, 15: 0.015, 16: 0.007, 17: 0.003, 18: 0.003}
limit P(corank 0) for uniform: 0.639
2x2 singular probability over F_3: 9 / 27 = 1/p: True


For uniform entries the corank law matches the limit (within sampling error). As $d$ decreases the corank grows, first through zero rows and repeated rows, then through subtler dependencies: Project 3 starts here.

**P12.4.** Encode a polynomial over $\mathbb F_q$ by its list of coefficients; multiplication of monic polynomials of degree $n$ gives monic polynomials of degree $2n$, and we count collisions with a dictionary.

In [5]:
from itertools import product
def monic(q, n):
    for c in product(range(q), repeat=n): yield tuple(c) + (1,)
def mul(a, b, q):
    r = [0] * (len(a) + len(b) - 1)
    for i, x in enumerate(a):
        if x:
            for j, y in enumerate(b): r[i+j] = (r[i+j] + x*y) % q
    return tuple(r)
def fourth(q, n):
    P = list(monic(q, n)); c = Counter(mul(a, b, q) for a in P for b in P)
    return sum(v*v for v in c.values())
for q, nmax in [(2, 8), (3, 5)]:
    for n in range(1, nmax + 1):
        F = fourth(q, n)
        print(f'q={q} n={n}: count {F:>12d}   / (n q^(2n)) = {F / (n * q**(2*n)):.4f}')
def squarefree(q, n):
    # g is squarefree iff no a^2 divides g for monic a of degree 1..n/2; count by exclusion over all products a^2 b
    nonsq = set(mul(mul(a, a, q), b, q) for k in range(1, n//2 + 1) for a in monic(q, k) for b in monic(q, n - 2*k))
    return q**n - len(nonsq)
print('squarefree monic counts, q=3:', [(n, squarefree(3, n), 3**n - 3**(n-1)) for n in range(2, 7)])

q=2 n=1: count            6   / (n q^(2n)) = 1.5000
q=2 n=2: count           32   / (n q^(2n)) = 1.0000
q=2 n=3: count          160   / (n q^(2n)) = 0.8333
q=2 n=4: count          768   / (n q^(2n)) = 0.7500
q=2 n=5: count         3584   / (n q^(2n)) = 0.7000
q=2 n=6: count        16384   / (n q^(2n)) = 0.6667
q=2 n=7: count        73728   / (n q^(2n)) = 0.6429


q=2 n=8: count       327680   / (n q^(2n)) = 0.6250
q=3 n=1: count           15   / (n q^(2n)) = 1.6667
q=3 n=2: count          189   / (n q^(2n)) = 1.1667
q=3 n=3: count         2187   / (n q^(2n)) = 1.0000
q=3 n=4: count        24057   / (n q^(2n)) = 0.9167
q=3 n=5: count       255879   / (n q^(2n)) = 0.8667
squarefree monic counts, q=3: [(2, 6, 6), (3, 18, 18), (4, 54, 54), (5, 162, 162), (6, 486, 486)]


The ratios decrease by exactly $\frac12$ (for $q=2$) and $\frac23$ (for $q=3$) times $\frac1n$ per step: every count above equals $q^{2n}\bigl((1-\frac1q)n+1\bigr)$. That is a conjecture suggested by the data, with $1-\frac1q=1/\zeta_q(2)$ playing the role of the constant; proving it (with the generating function of the chapter) is the first step of Project 4. The squarefree counts agree with Exercise 12.4: $q^n-q^{n-1}$.